# 05 · Fine-tune a small model with the `Trainer`

You will fine-tune DistilBERT into a 3-class **support-ticket intent classifier** (billing / technical / account) with the Hugging Face `Trainer`, on CPU, in well under a minute. Along the way: a new classification head, tokenising with `datasets.map`, dynamic padding, every `TrainingArguments` knob you'll be asked about, metrics, loss curves, a confusion matrix, saving and reloading, and how the `Trainer` compares with a plain PyTorch loop and TRL's `SFTTrainer`.

**Time:** ~25 min · **Runs:** offline on CPU in about a minute · **Needs:** [03 · datasets](03_datasets_library.ipynb); [training loops and optimizers](../06_deep_learning/02_training_loop_optimizers_lr_schedules.ipynb) for what happens inside

## Why this matters in interviews

- "Fine-tune vs prompt vs RAG" is only convincing if you have actually fine-tuned something and know what the knobs do: learning rate, epochs, batch size, weight decay, evaluation strategy.
- A small fine-tuned classifier is often the right production answer for routing, intent detection and moderation: milliseconds per call on CPU and no per-token bill (`tn39`).
- Interviewers probe overfitting ("train loss falls, eval loss rises"), metric choice (accuracy vs macro-F1 on imbalanced classes), and reproducibility (seeds, saved configs).

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `tn03` | What is fine-tuning, and what are its main types? |
| `fo33` | What is transfer learning, and how does it relate to fine-tuning? |
| `tn22` | What is the difference between an epoch, a step and a batch in LLM training? |
| `tn28` | What is overfitting in the context of fine-tuning an LLM, and how do you spot it? |
| `tn20` | How do you evaluate a fine-tuned model before shipping it? |
| `tn39` | How do you decide between fine-tuning a small model and prompting a large one? |
| `tn13` | How much data do you need to fine-tune, and what does good data look like? |

In [ ]:
import time, tempfile, os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
import transformers, datasets
from datasets import Dataset, ClassLabel
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer,
                          DataCollatorWithPadding, pipeline, set_seed)
from transformers.trainer_callback import PrinterCallback
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

transformers.utils.logging.set_verbosity_error()
transformers.utils.logging.disable_progress_bar()
datasets.disable_progress_bars()
set_seed(0)                       # seeds python, numpy and torch in one call
print("transformers", transformers.__version__, "| torch", torch.__version__)

## 1. The data: 60 tickets, 3 intents

Tiny on purpose, so it trains on a laptop in seconds. A real intent classifier would want a few hundred examples per class, collected from real traffic (`tn13`). The classes are balanced here; section 5 says what changes when they aren't.

In [ ]:
billing = ["I was charged twice this month", "Why is my invoice so high", "Refund my last payment please",
           "My card was billed without permission", "How do I update my payment method", "The subscription price went up",
           "I need a copy of my receipt", "Cancel the extra charge on my bill", "Can I pay by bank transfer",
           "You billed me after I cancelled", "What is this fee on my statement", "I want a refund for the annual plan",
           "Payment failed but money left my account", "Is VAT included in the price", "Change my plan to monthly billing",
           "Why was I charged a late fee", "My discount code was not applied", "Send me last year's invoices",
           "Do you offer a student discount", "The price on the invoice is wrong"]
technical = ["The app crashes when I open it", "I get error 500 on the dashboard", "The page will not load on Chrome",
             "Sync is not working on my phone", "The export button does nothing", "My upload keeps failing at 99 percent",
             "Notifications stopped arriving", "The API returns a timeout", "Dark mode breaks the layout",
             "Search results are empty for every query", "The website is very slow today",
             "I cannot connect the printer integration", "Video calls freeze after a minute",
             "The installer says file corrupted", "Login page shows a blank screen",
             "The chart does not update with new data", "Webhooks are not firing", "Import from CSV throws an error",
             "The mobile app drains my battery", "Emails from the app go to spam"]
account = ["How do I reset my password", "I want to delete my account", "Change the email on my profile",
           "Add a new user to my team", "I am locked out of my account", "Enable two factor authentication",
           "How do I change my username", "Transfer ownership to my colleague", "Remove my old phone number",
           "I did not get the verification email", "Merge my two accounts", "Update my company name on the profile",
           "Who has admin rights on our workspace", "Deactivate a former employee", "Recover my account after a hack",
           "Set up single sign on", "I forgot which email I signed up with", "Change my display language",
           "Download all my personal data", "Log me out of all devices"]
LABELS = ["billing", "technical", "account"]
id2label = dict(enumerate(LABELS)); label2id = {v: k for k, v in id2label.items()}

ds = Dataset.from_dict({"text": billing + technical + account, "label": [0] * 20 + [1] * 20 + [2] * 20})
ds = ds.cast_column("label", ClassLabel(names=LABELS))
splits = ds.train_test_split(test_size=0.25, seed=0, stratify_by_column="label")
print(splits)
print("test examples per class:", np.bincount(splits["test"]["label"]))

## 2. Start from a pretrained model, with a new head

**In plain English:** we don't train from scratch. DistilBERT already understands English from pretraining (and this checkpoint was then fine-tuned for sentiment). We keep that body and replace the final layer, which has 2 outputs (negative/positive), with a fresh one that has 3 outputs (our intents). That is **transfer learning** (`fo33`).

`num_labels=3` asks for the new head. `ignore_mismatched_sizes=True` tells transformers it's OK that the checkpoint's 2-way head doesn't fit, so it initialises the head randomly instead of raising an error.

In [ ]:
BASE = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"
tok = AutoTokenizer.from_pretrained(BASE)
model = AutoModelForSequenceClassification.from_pretrained(
    BASE, num_labels=3, id2label=id2label, label2id=label2id, ignore_mismatched_sizes=True)
print("new head:", model.classifier, "| labels:", model.config.id2label)
assert model.classifier.out_features == 3

def tokenize(batch):
    return tok(batch["text"], truncation=True, max_length=32)       # tickets are short: 32 tokens is plenty

encoded = splits.map(tokenize, batched=True, remove_columns=["text"])
lengths = [len(x) for x in encoded["train"]["input_ids"]]
print(f"token lengths: min {min(lengths)}, median {int(np.median(lengths))}, max {max(lengths)} (limit 32)")
print("columns the Trainer will see:", encoded["train"].column_names)

Before training, the new head is random, so the model is guessing. Check it (and keep the number for comparison):

In [ ]:
def evaluate_raw(m, split):
    m.eval()
    batch = tok(list(splits[split]["text"]), padding=True, return_tensors="pt")
    with torch.no_grad():
        preds = m(**batch).logits.argmax(-1).numpy()
    return accuracy_score(splits[split]["label"], preds)

acc_before = evaluate_raw(model, "test")
print(f"test accuracy with the untrained head: {acc_before:.2f} (chance = 0.33)")

### Freezing the lower layers (and why)

Full fine-tuning updates all 67 M parameters. Here we **freeze the embeddings and the first 4 of the 6 transformer layers**, and train only the top 2 layers plus the head. The reasons:

- **Speed on CPU:** frozen layers need no gradients and no optimizer state, so backpropagation stops early. That keeps this notebook fast.
- **Tiny data:** with 45 examples, updating fewer weights overfits less. Lower layers hold general language features we want to keep, while the top layers are the most task-specific.
- It is the classic middle ground between "train only the head" (fast, but often underfits) and full fine-tuning. **LoRA** is the modern version of the same idea for LLMs: freeze everything and train small added matrices ([LoRA with PEFT](../11_finetuning/04_lora_with_peft.ipynb)).

In [ ]:
for p in model.distilbert.embeddings.parameters():
    p.requires_grad = False
for layer in model.distilbert.transformer.layer[:4]:
    for p in layer.parameters():
        p.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"trainable parameters: {trainable:,} of {total:,} ({trainable / total:.0%})")
assert 0.1 < trainable / total < 0.3

## 3. `TrainingArguments`: every knob, explained

| Argument | Value | What it does |
|---|---|---|
| `output_dir` | a temp folder | where checkpoints and logs would go |
| `learning_rate` | `2e-4` | step size for AdamW. Full fine-tuning of BERT-size models usually uses 2e-5–5e-5; training fewer layers tolerates a higher rate |
| `per_device_train_batch_size` | `8` | examples per step. With 45 examples that gives 6 **steps** per **epoch** (`tn22`) |
| `num_train_epochs` | `4` | full passes over the training data; small data needs a few, but too many memorises it |
| `weight_decay` | `0.01` | AdamW's decoupled L2 pull towards zero, a mild regulariser |
| `eval_strategy` | `"epoch"` | run the eval set after every epoch, so we can see overfitting. (It was called `evaluation_strategy` in older versions.) |
| `logging_steps` | `2` | record the training loss every 2 steps into `trainer.state.log_history` |
| `save_strategy` | `"no"` | don't write checkpoints (we save once at the end). In real runs use `"epoch"` with `load_best_model_at_end=True` |
| `report_to` | `"none"` | no Weights & Biases / TensorBoard integration |
| `use_cpu` | `True` | force CPU even if a GPU exists (it replaced the old `no_cuda`) |
| `seed` | `0` | seeds the head initialisation, the shuffling and dropout, so reruns match |
| `lr_scheduler_type` (default) | `"linear"` | the learning rate decays linearly to 0; add `warmup_steps` for larger runs |

Other arguments that come up on a GPU: `bf16=True` (mixed precision), `gradient_accumulation_steps` (a large effective batch on a small GPU), `gradient_checkpointing=True` (trade compute for memory), and `load_best_model_at_end` + `metric_for_best_model` (early-stopping style).

In [ ]:
def compute_metrics(eval_pred):
    """Called by the Trainer after each evaluation: logits + labels -> a dict of metrics."""
    preds = eval_pred.predictions.argmax(axis=-1)
    return {"accuracy": accuracy_score(eval_pred.label_ids, preds),
            "f1_macro": f1_score(eval_pred.label_ids, preds, average="macro")}

args = TrainingArguments(
    output_dir=tempfile.mkdtemp(),
    learning_rate=2e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    num_train_epochs=4,
    weight_decay=0.01,
    eval_strategy="epoch",
    logging_steps=2,
    save_strategy="no",
    report_to="none",
    use_cpu=True,
    seed=0,
    disable_tqdm=True,
)
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=encoded["train"],
    eval_dataset=encoded["test"],
    processing_class=tok,                        # v5 name; older code passed tokenizer=tok
    data_collator=DataCollatorWithPadding(tok),  # pads each batch to its own longest row
    compute_metrics=compute_metrics,
)
trainer.remove_callback(PrinterCallback)         # keep the output tidy; the logs stay in trainer.state
print("steps per epoch:", len(trainer.get_train_dataloader()))

## 4. Train and watch the curves

In [ ]:
t0 = time.perf_counter()
train_out = trainer.train()
print(f"trained {train_out.global_step} steps in {time.perf_counter() - t0:.1f}s, "
      f"mean training loss {train_out.training_loss:.3f}")

history = trainer.state.log_history
train_pts = [(h["epoch"], h["loss"]) for h in history if "loss" in h]
eval_pts = [(h["epoch"], h["eval_loss"], h["eval_accuracy"]) for h in history if "eval_loss" in h]
for e, l, a in eval_pts:
    print(f"epoch {e:.0f}: eval_loss {l:.3f}  eval_accuracy {a:.2f}")

fig, ax = plt.subplots(figsize=(8, 3.8))
ax.plot(*zip(*train_pts), "o-", label="train loss (every 2 steps)")
ax.plot([e for e, _, _ in eval_pts], [l for _, l, _ in eval_pts], "s--", ms=8, label="eval loss (every epoch)")
ax.set_xlabel("epoch"); ax.set_ylabel("cross-entropy loss"); ax.set_title("Fine-tuning DistilBERT on 45 tickets")
ax.legend(); plt.show()
assert eval_pts[-1][1] < eval_pts[0][1]                 # eval loss went down: it learned, not just memorised

How to read the plot:

- **Both losses fall.** The model is learning something that transfers to tickets it never saw. The training loss is noisy because each point averages only 2 small batches.
- **The overfitting signature** (`tn28`) would be training loss still falling while eval loss turns *up*. With more epochs on 45 examples you would eventually see it. Then you stop at the best epoch (`load_best_model_at_end=True`), add data, or train fewer parameters.
- With 15 test examples, one mistake is ~7 points of accuracy. **Small eval sets are noisy**, so don't celebrate a 0.93 vs 0.87 difference.

## 5. Evaluate properly

In [ ]:
metrics = trainer.evaluate()
print({k: round(v, 3) for k, v in metrics.items() if k in ("eval_loss", "eval_accuracy", "eval_f1_macro")})
assert metrics["eval_accuracy"] > max(acc_before, 0.6)

pred = trainer.predict(encoded["test"])
y_true, y_pred = pred.label_ids, pred.predictions.argmax(-1)
cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])

fig, ax = plt.subplots(figsize=(4.8, 4))
ax.imshow(cm, cmap="Blues")
for i in range(3):
    for j in range(3):
        ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black")
ax.set_xticks(range(3), LABELS); ax.set_yticks(range(3), LABELS)
ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title("Confusion matrix (test set)")
plt.show()

for text, t, p in zip(splits["test"]["text"], y_true, y_pred):
    if t != p:
        print(f"MISTAKE: {text!r}: true {LABELS[t]}, predicted {LABELS[p]}")

Accuracy is fine for balanced classes. With imbalanced classes (say 90 % billing), a model that always says "billing" gets 0.9 accuracy, and **macro-F1** (the average of per-class F1) exposes it. The confusion matrix shows *which* intents get mixed up, and reading the actual mistakes is the fastest way to find label noise or a missing class (`tn20`). Before shipping you would also want a bigger held-out set drawn from real traffic, a comparison against the baseline you're replacing (rules, or a prompted LLM), and latency and cost numbers.

## 6. Use it: predict on new tickets

In [ ]:
clf = pipeline("text-classification", model=trainer.model, tokenizer=tok, device="cpu")
new_tickets = ["My credit card was charged three times", "The dashboard shows error 404",
               "Please add my colleague to the workspace", "Where can I download my invoice?"]
preds_new = clf(new_tickets)
for t, p in zip(new_tickets, preds_new):
    print(f"{p['label']:>9s} {p['score']:.2f}  {t}")
assert [p["label"] for p in preds_new[:3]] == ["billing", "technical", "account"]

The first three are right. The invoice question is a genuine error: it goes to `account` with a noticeably lower score. With only 45 training examples the model picks up word cues ("download" appears in an account example, "Download all my personal data") more than the intent. More varied data per class fixes this kind of error better than more epochs do, and the lower score is a hint that routing on confidence (exercise 3) will help.

## 7. Save, reload, share

`save_pretrained` writes `config.json` (with our `id2label`, so the label names travel with the model) and `model.safetensors`. Save the tokenizer next to it, so the folder is self-contained and loads with `from_pretrained(folder)` or `pipeline(model=folder)`.

In [ ]:
out_dir = Path(tempfile.mkdtemp()) / "intent-classifier"
trainer.save_model(str(out_dir))            # = model.save_pretrained + tokenizer.save_pretrained (processing_class)
print("saved:", sorted(p.name for p in out_dir.iterdir()))
print(f"model.safetensors: {(out_dir / 'model.safetensors').stat().st_size / 1e6:.0f} MB")

reloaded = pipeline("text-classification", model=str(out_dir), device="cpu")
again = reloaded(new_tickets)
assert [p["label"] for p in again] == [p["label"] for p in preds_new]
assert np.allclose([p["score"] for p in again], [p["score"] for p in preds_new], atol=1e-5)
print("reloaded from disk -> identical predictions")

Note `training_args.bin`: it is a **pickle** of the `TrainingArguments`, kept for reproducibility and not needed for inference. As [notebook 01](01_hub_and_pipelines.ipynb) showed, never unpickle one from an untrusted repo.

To share it on the Hub (needs a write token, `HF_TOKEN`; not run here):

```python
trainer.push_to_hub("your-name/intent-classifier", private=True)   # uploads the model, tokenizer and an auto-written model card
# or: model.push_to_hub("your-name/intent-classifier"); tok.push_to_hub("your-name/intent-classifier")
# or set TrainingArguments(push_to_hub=True, hub_model_id="your-name/intent-classifier") to push during training
```

## 8. Trainer vs a plain PyTorch loop vs TRL's `SFTTrainer`

| | Plain PyTorch loop | 🤗 `Trainer` | TRL `SFTTrainer` (and `DPOTrainer`, `GRPOTrainer`) |
|---|---|---|---|
| You write | the loop: batches, forward, `loss.backward()`, `optimizer.step()`, scheduler, eval, logging, saving | datasets, `TrainingArguments`, optional `compute_metrics` | a dataset in `messages` / `prompt`-`completion` form, plus `SFTConfig` |
| Built in | nothing | AdamW + LR schedule, eval/save strategies, mixed precision, gradient accumulation, multi-GPU/DeepSpeed/FSDP via accelerate, logging integrations, resume from checkpoint | everything in `Trainer`, **plus** chat templates, assistant-only loss masking, packing, PEFT/LoRA and QLoRA wiring |
| Best for | research code, unusual losses, learning how it works | classification, token tagging, standard fine-tunes of encoders | instruction-tuning / SFT of LLMs, then preference tuning |
| Risk | bugs in the parts you wrote (forgetting `model.eval()` or `zero_grad`) | silent defaults (for example, it drops columns the model doesn't accept) | the same defaults as `Trainer`, plus chat-template and masking details |

What `trainer.train()` does is the loop you would write yourself (exercise 2 writes one):

```
for epoch in range(num_train_epochs):
    for batch in DataLoader(train_dataset, batch_size, shuffle=True, collate_fn=data_collator):
        loss = model(**batch).loss          # the model computes cross-entropy because `labels` is in the batch
        loss.backward(); optimizer.step(); scheduler.step(); optimizer.zero_grad()
    if eval_strategy == "epoch": evaluate() -> compute_metrics()
```

## Try it yourself

**1. Per-class report.** Use `sklearn.metrics.classification_report` on `y_true`/`y_pred` with the label names. Which class has the lowest recall?

In [ ]:
# Solution — try it yourself first, then run this
from sklearn.metrics import classification_report, recall_score
print(classification_report(y_true, y_pred, labels=[0, 1, 2], target_names=LABELS, zero_division=0))
recalls = recall_score(y_true, y_pred, labels=[0, 1, 2], average=None, zero_division=0)
print("lowest recall:", LABELS[int(np.argmin(recalls))], f"({recalls.min():.2f})")

**2. The same training as a plain PyTorch loop.** Train a *fresh* copy of the model (same freezing) for 2 epochs with your own loop: `DataLoader` + `DataCollatorWithPadding`, AdamW, `loss.backward()`. Then report test accuracy. It should beat the untrained head clearly.

In [ ]:
# Solution — try it yourself first, then run this
from torch.utils.data import DataLoader
set_seed(0)
m2 = AutoModelForSequenceClassification.from_pretrained(BASE, num_labels=3, id2label=id2label, label2id=label2id,
                                                        ignore_mismatched_sizes=True)
for p in list(m2.distilbert.embeddings.parameters()) + [p for l in m2.distilbert.transformer.layer[:4] for p in l.parameters()]:
    p.requires_grad = False
loader = DataLoader(encoded["train"].rename_column("label", "labels"), batch_size=8, shuffle=True,
                    collate_fn=DataCollatorWithPadding(tok), generator=torch.Generator().manual_seed(0))
opt = torch.optim.AdamW([p for p in m2.parameters() if p.requires_grad], lr=2e-4, weight_decay=0.01)
m2.train()
for epoch in range(2):
    losses = []
    for batch in loader:
        loss = m2(**batch).loss           # labels in the batch -> the model returns cross-entropy
        loss.backward(); opt.step(); opt.zero_grad()
        losses.append(loss.item())
    print(f"epoch {epoch}: mean train loss {np.mean(losses):.3f}")
acc_loop = evaluate_raw(m2, "test")       # evaluate_raw switches to eval mode (dropout off)
print(f"plain-loop test accuracy after 2 epochs: {acc_loop:.2f} (untrained head: {acc_before:.2f})")
assert acc_loop > acc_before

**3. Confidence-based routing.** In production you'd send low-confidence tickets to a human. Using `clf(..., top_k=None)`, route a ticket to `"human"` when the top score is below 0.6. Try an off-topic ticket such as "What's the weather like in Paris?". Does the model know that it doesn't know?

In [ ]:
# Solution — try it yourself first, then run this
def route(text, threshold=0.6):
    scores = clf(text, top_k=None)
    best = max(scores, key=lambda d: d["score"])
    return (best["label"] if best["score"] >= threshold else "human"), round(best["score"], 2)

for t in ["My card was charged twice", "What's the weather like in Paris?", "I love your product!"]:
    print(f"{str(route(t)):22s} <- {t}")
# A 3-way softmax must spread 100% over billing/technical/account, so off-topic text can still get a
# confident label. A threshold catches some of these; an explicit "other" class trained on examples is better.

## Say it in an interview

- **30-second answer:** "I start from a pretrained encoder, swap the head for my label count (`num_labels`, `ignore_mismatched_sizes=True`), tokenise with `datasets.map`, pad dynamically with `DataCollatorWithPadding`, and train with the `Trainer` using a learning rate around 2e-5–5e-5 for full fine-tuning, 2–4 epochs, weight decay 0.01, and evaluation every epoch with accuracy and macro-F1. I watch train vs eval loss for overfitting, read the confusion matrix and the actual mistakes, and save with `save_pretrained` so `id2label` travels with the model."
- **Terms (`tn22`):** batch = examples per step; step = one optimizer update; epoch = one full pass. Steps per epoch = ceil(N / batch size). Effective batch = batch × gradient accumulation × devices.
- **Numbers:** a few hundred labelled examples per class usually beat prompting on a narrow classification task, at ~10 ms per call on CPU and no token bill. DistilBERT is 67 M parameters (~270 MB in fp32).
- **Trade-offs:** freezing layers (or LoRA) is faster and overfits less on tiny data; full fine-tuning is best with plenty of data. Fine-tune a small model when the task is narrow and high-volume; prompt a large model when the task is broad, changes often, or has no labels yet.
- **Traps:** evaluating on training data; a split that isn't stratified on imbalanced labels; accuracy on imbalanced classes; forgetting `id2label` (you ship `LABEL_0`); a softmax classifier has no "none of the above", so add an explicit class; renamed v5 arguments (`eval_strategy`, `processing_class`, `use_cpu`).

## Next

- LLM fine-tuning: [when to fine-tune and data prep](../11_finetuning/01_when_to_fine_tune_and_data_prep.ipynb) · [SFT from scratch with loss masking](../11_finetuning/02_sft_from_scratch_with_loss_masking.ipynb) · [LoRA with PEFT](../11_finetuning/04_lora_with_peft.ipynb) · [QLoRA](../11_finetuning/05_qlora_with_bitsandbytes_and_peft.ipynb)
- Metrics in depth: [confusion matrix, ROC and PR](../05_machine_learning/04_metrics_confusion_matrix_roc_pr.ipynb) · [overfitting and cross-validation](../05_machine_learning/05_overfitting_regularization_cross_validation.ipynb)
- Make it smaller: [06 · Quantization with bitsandbytes](06_quantization_with_bitsandbytes.ipynb)
- Theory: [genai_flow course](../../genai_flow/index.html), [interview bank](../../ai_interview_prep/index.html)